<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/evaluation/06_CREPE_VRAM_benchmark_ipynb%EC%9D%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# Phase 8: E2E Bass Transcription Pipeline Benchmark (Colab)
# [Cell 1] 환경 설정, GPU 할당 검증, 저장소 동기화 및 무결성 점검
# ==============================================================================

import os
import sys
import shutil
import importlib.util
import subprocess
from google.colab import drive
import torch
from datetime import datetime

# [사용자 환경 변수 세팅] - 본인의 드라이브 경로로 맞춰주세요.
DRIVE_DATASET_DIR = "/content/drive/MyDrive/Bass_separator/dataset"
DRIVE_ZIP_PATH = f"{DRIVE_DATASET_DIR}/slakh_test.zip"

print("📂 구글 드라이브 마운트 중...")
drive.mount('/content/drive')

if not torch.cuda.is_available():
    raise SystemError("❌ GPU가 할당되지 않았습니다. 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 T4 GPU를 선택하십시오.")
print(f"✅ GPU 활성화됨: {torch.cuda.get_device_name(0)}")

print("\n📦 저장소 클론 및 작업 공간 초기화 중...")
# 안전 지대로 이동
%cd /content
!rm -rf /content/Bass-separator
!git clone https://github.com/sjkim-audio/Bass-separator.git /content/Bass-separator

if "/content/Bass-separator" not in sys.path:
    sys.path.append("/content/Bass-separator")
%cd /content/Bass-separator

print("\n🔧 [시스템] 필수 도구 확인 중...")
if shutil.which("ffmpeg") is None:
    print("⚠️ FFmpeg가 없습니다. apt-get으로 설치합니다...")
    !apt-get update -qq
    !apt-get install -y ffmpeg -qq
else:
    print("✅ FFmpeg가 이미 설치되어 있습니다.")

print("\n🐍 [파이썬] 라이브러리 설치 중...")
!pip install -q -r requirements.txt
!pip install -q mir_eval museval pretty_midi

print("\n🏥 설치 무결성 점검 (Health Check)...")
critical_libs = ["demucs", "torchaudio", "librosa", "museval", "mir_eval"]
missing = [lib for lib in critical_libs if importlib.util.find_spec(lib) is None]

if not missing:
    print("✅ 필수 라이브러리가 모두 정상적으로 준비되었습니다!")
else:
    raise ImportError(f"❌ 다음 라이브러리가 누락되었습니다: {', '.join(missing)}")

📂 구글 드라이브 마운트 중...
Mounted at /content/drive
✅ GPU 활성화됨: Tesla T4

📦 저장소 클론 및 작업 공간 초기화 중...
/content
Cloning into '/content/Bass-separator'...
remote: Enumerating objects: 2628, done.
remote: Counting objects: 100% (397/397), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 2628 (delta 337), reused 268 (delta 268), pack-reused 2231 (from 2)
Receiving objects: 100% (2628/2628), 305.58 MiB | 17.30 MiB/s, done.
Resolving deltas: 100% (1623/1623), done.
/content/Bass-separator

🔧 [시스템] 필수 도구 확인 중...
✅ FFmpeg가 이미 설치되어 있습니다.

🐍 [파이썬] 라이브러리 설치 중...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# ==============================================================================
# [Cell 2] 벤치마크 스크립트 파일 생성
# ==============================================================================
%%writefile benchmark.py
import time
import numpy as np
import torch
from src.transcription.tracker import get_f0_crepe_robust

def run_benchmark():
    print("⏳ 더미 오디오 데이터(30분 길이) 생성 중...")
    sr = 16000
    duration_sec = 1800
    dummy_audio = np.random.randn(sr * duration_sec).astype(np.float32)

    print("🔥 GPU 워밍업 중...")
    _ = get_f0_crepe_robust(
        audio=dummy_audio[:sr*2],
        sr=sr, hop_length=160, model_capacity='full', batch_size=512
    )
    torch.cuda.synchronize()

    print("🚀 벤치마크 측정 시작...")
    start_time = time.perf_counter()

    f0, conf, onset = get_f0_crepe_robust(
        audio=dummy_audio,
        sr=sr, hop_length=160, model_capacity='full', batch_size=512
    )
    torch.cuda.synchronize()

    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    print(f"⏱️ 순수 피치 트래킹 소요 시간: {elapsed_time:.2f} 초")
    print(f"📊 오디오 1초당 처리 속도 (RTF): {duration_sec / elapsed_time:.2f} 배속")

if __name__ == "__main__":
    run_benchmark()

Overwriting benchmark.py


In [5]:
# ==============================================================================
# [Cell 3] 캐시 클리어 제거 전/후 성능 자동 비교 (A/B 테스트)
# ==============================================================================
!echo "🟢 === [최적화 후] 현재 코드 성능 측정 ==="
!python benchmark.py

!echo -e "\n🔴 === [최적화 전] 과거 코드 성능 측정 ==="
# 이전 커밋(empty_cache가 있던 상태)으로 되돌리기
#
!git checkout HEAD~2 --quiet
!python benchmark.py

!echo -e "\n🔄 === 원래 브랜치로 복귀 ==="
!git checkout main --quiet
!echo "✅ 벤치마크 테스트 완료 및 코드 원상복구 성공"

🟢 === [최적화 후] 현재 코드 성능 측정 ===
⏳ 더미 오디오 데이터(30분 길이) 생성 중...
🔥 GPU 워밍업 중...
/usr/local/lib/python3.13/dist-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
🚀 벤치마크 측정 시작...
⏱️ 순수 피치 트래킹 소요 시간: 148.68 초
📊 오디오 1초당 처리 속도 (RTF): 12.11 배속

🔴 === [최적화 전] 과거 코드 성능 측정 ===
⏳ 더미 오디오 데이터(30분 길이) 생성 중...
🔥 GPU 워밍업 중...
/usr/local/lib/python3.13/dist-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
🚀 벤치마크 측정 시작...
⏱️ 순수 피치 트래킹 소요 시간: 154.82 초
📊 오디오 1초당 처리 속도 (RTF): 11.63 배속

🔄 === 원래 브랜치로 복귀 ===
✅ 벤치마크 테스트 완료 및 코드 원상복구 성공
